# Multimodal AI and Vision-Language Models
## Contrastive Learning, Cross-Modal Embeddings, and Fusion Strategies

Hands-on exploration using real embedding models and pretrained vision-language models.

**Prerequisites:**
```bash
pip install sentence-transformers torch torchvision matplotlib numpy pandas scikit-learn pillow scipy
```

📺 **Video Lecture:** [https://youtu.be/GEmTIyLfVLU](https://youtu.be/GEmTIyLfVLU)

## Cell 1: Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torchvision
from torchvision import transforms, models
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax
from PIL import Image, ImageDraw
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Section 1: Creating Multimodal Data with Real Images and Text

In [ ]:
# Create synthetic images with PIL (real image data)
# Each concept is represented by a colored shape

concepts = {
    'red_circle': {'color': (255, 0, 0), 'shape': 'circle', 'desc': 'A bright red circle on white background'},
    'blue_square': {'color': (0, 0, 255), 'shape': 'square', 'desc': 'A bright blue square on white background'},
    'green_triangle': {'color': (0, 128, 0), 'shape': 'triangle', 'desc': 'A green triangle on white background'},
    'yellow_pentagon': {'color': (255, 255, 0), 'shape': 'pentagon', 'desc': 'A yellow pentagon on white background'},
    'purple_star': {'color': (128, 0, 128), 'shape': 'star', 'desc': 'A purple star on white background'},
    'orange_circle': {'color': (255, 165, 0), 'shape': 'circle', 'desc': 'An orange circle on white background'},
    'cyan_square': {'color': (0, 255, 255), 'shape': 'square', 'desc': 'A cyan square on white background'},
    'pink_diamond': {'color': (255, 192, 203), 'shape': 'diamond', 'desc': 'A pink diamond on white background'},
    'brown_rectangle': {'color': (165, 42, 42), 'shape': 'rectangle', 'desc': 'A brown rectangle on white background'},
    'gray_hexagon': {'color': (128, 128, 128), 'shape': 'hexagon', 'desc': 'A gray hexagon on white background'},
}

# Generate images
images = {}
texts = {}

def draw_shape(img, color, shape, position=(112, 112), size=80):
    """Draw a shape on the image."""
    draw = ImageDraw.Draw(img)
    x, y = position
    
    if shape == 'circle':
        draw.ellipse([x - size, y - size, x + size, y + size], fill=color, outline=color)
    elif shape == 'square':
        draw.rectangle([x - size, y - size, x + size, y + size], fill=color, outline=color)
    elif shape == 'triangle':
        draw.polygon([(x, y - size), (x + size, y + size), (x - size, y + size)], fill=color, outline=color)
    elif shape == 'rectangle':
        draw.rectangle([x - size, y - size//2, x + size, y + size//2], fill=color, outline=color)
    elif shape == 'pentagon':
        points = [(x, y - size), (x + size, y - size//2), (x + size//2, y + size), 
                  (x - size//2, y + size), (x - size, y - size//2)]
        draw.polygon(points, fill=color, outline=color)
    elif shape == 'star':
        points = [(x, y - size), (x + size//2, y - size//3), (x + size, y - size//3),
                  (x + size//2, y + size//3), (x + size, y + size), (x, y + size//2),
                  (x - size, y + size), (x - size//2, y + size//3), (x - size, y - size//3),
                  (x - size//2, y - size//3)]
        draw.polygon(points, fill=color, outline=color)
    elif shape == 'diamond':
        points = [(x, y - size), (x + size, y), (x, y + size), (x - size, y)]
        draw.polygon(points, fill=color, outline=color)
    elif shape == 'hexagon':
        points = [(x + size, y), (x + size//2, y - size), (x - size//2, y - size),
                  (x - size, y), (x - size//2, y + size), (x + size//2, y + size)]
        draw.polygon(points, fill=color, outline=color)

# Create images with variations
for concept_name, props in concepts.items():
    for i in range(3):  # 3 variations per concept (slightly different positions/sizes)
        img = Image.new('RGB', (224, 224), 'white')
        pos = (112 + np.random.randint(-20, 21), 112 + np.random.randint(-20, 21))
        size = 80 + np.random.randint(-10, 11)
        draw_shape(img, props['color'], props['shape'], position=pos, size=size)
        images[f"{concept_name}_{i}"] = img
        texts[f"{concept_name}_{i}"] = props['desc']

print(f"Created {len(images)} images and {len(texts)} text descriptions")
print(f"Sample concepts: {list(concepts.keys())[:3]}")

# Display a few sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for idx, (name, img) in enumerate(list(images.items())[:10]):
    ax = axes[idx // 5, idx % 5]
    ax.imshow(img)
    ax.set_title(name.split('_')[0] + '_' + name.split('_')[1][:3], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Generated Images', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## Section 2: Real Text Embeddings with Sentence-Transformers

In [ ]:
# Load a real sentence embedding model
print("Loading SentenceTransformer model...")
text_model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Model device: {text_model.device}")

# Generate embeddings for all texts
text_descriptions = list(texts.values())
text_embeddings = text_model.encode(text_descriptions, convert_to_tensor=True)

# Convert to numpy
text_embeddings_np = text_embeddings.cpu().numpy()

# Normalize
text_embeddings_np = text_embeddings_np / np.linalg.norm(text_embeddings_np, axis=1, keepdims=True)

print(f"Text embeddings shape: {text_embeddings_np.shape}")
print(f"Embedding dimension: {text_embeddings_np.shape[1]}")
print(f"Sample text: {text_descriptions[0]}")
print(f"Sample embedding norm: {np.linalg.norm(text_embeddings_np[0]):.4f}")

## Section 3: Real Image Embeddings with ResNet

In [ ]:
# Load pretrained ResNet18 and remove classification head
print("Loading ResNet18 pretrained on ImageNet...")
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet_encoder = torch.nn.Sequential(*list(resnet.children())[:-1])  # Remove FC layer
resnet_encoder.to(device)
resnet_encoder.eval()

# Preprocessing pipeline
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Generate image embeddings
image_embeddings_list = []
image_names = []

with torch.no_grad():
    for name, img in images.items():
        tensor = preprocess(img).unsqueeze(0).to(device)
        emb = resnet_encoder(tensor).squeeze().cpu().numpy()
        # Flatten if needed
        emb = emb.flatten() if emb.ndim > 1 else emb
        image_embeddings_list.append(emb)
        image_names.append(name)

image_embeddings_np = np.array(image_embeddings_list)
# Normalize
image_embeddings_np = image_embeddings_np / np.linalg.norm(image_embeddings_np, axis=1, keepdims=True)

print(f"Image embeddings shape: {image_embeddings_np.shape}")
print(f"Embedding dimension: {image_embeddings_np.shape[1]}")
print(f"Total samples: {len(image_embeddings_np) + len(text_embeddings_np)}")

## Section 4: Addressing the Modality Gap with Linear Projection

In [ ]:
# The image encoder outputs 512 dims, text encoder outputs 384 dims
# Project both to a common dimension
shared_dim = 256

print(f"Image embedding dimension: {image_embeddings_np.shape[1]}")
print(f"Text embedding dimension: {text_embeddings_np.shape[1]}")
print(f"Projecting both to shared dimension: {shared_dim}")

# Simple linear projection matrices (in practice, would be learned)
proj_image = np.random.randn(image_embeddings_np.shape[1], shared_dim) * 0.01
proj_text = np.random.randn(text_embeddings_np.shape[1], shared_dim) * 0.01

# Project embeddings
image_embeddings_proj = image_embeddings_np @ proj_image
text_embeddings_proj = text_embeddings_np @ proj_text

# Normalize projected embeddings
image_embeddings_proj = image_embeddings_proj / np.linalg.norm(image_embeddings_proj, axis=1, keepdims=True)
text_embeddings_proj = text_embeddings_proj / np.linalg.norm(text_embeddings_proj, axis=1, keepdims=True)

print(f"Projected image embeddings shape: {image_embeddings_proj.shape}")
print(f"Projected text embeddings shape: {text_embeddings_proj.shape}")

## Section 5: Contrastive Loss (InfoNCE)

In [ ]:
def contrastive_loss(image_emb, text_emb, temperature=0.07):
    """
    Contrastive loss (InfoNCE / CLIP-style).
    Assumes image_emb[i] and text_emb[i] are paired.
    Maximizes similarity for matched pairs, minimizes for unmatched.
    """
    # Compute cosine similarity matrix
    logits = np.dot(image_emb, text_emb.T) / temperature
    
    batch_size = logits.shape[0]
    labels = np.arange(batch_size)
    
    # Cross-entropy loss
    # Image-to-text: for each image, find the correct text
    probs_img_txt = softmax(logits, axis=1)
    loss_img_txt = -np.log(probs_img_txt[np.arange(batch_size), labels] + 1e-8)
    
    # Text-to-image: for each text, find the correct image
    probs_txt_img = softmax(logits.T, axis=1)
    loss_txt_img = -np.log(probs_txt_img[np.arange(batch_size), labels] + 1e-8)
    
    # Average both directions
    total_loss = (loss_img_txt.mean() + loss_txt_img.mean()) / 2
    return total_loss, logits

# Compute loss on a subset
batch_size = 20
loss, logits = contrastive_loss(
    image_embeddings_proj[:batch_size], 
    text_embeddings_proj[:batch_size], 
    temperature=0.07
)

print(f"Contrastive Loss (InfoNCE): {loss:.4f}")
print(f"Logits shape: {logits.shape}")

# Accuracy on this batch (image-to-text retrieval)
predictions = np.argmax(logits, axis=1)
ground_truth = np.arange(batch_size)
accuracy = (predictions == ground_truth).mean()
print(f"Image-to-Text Matching Accuracy on batch: {accuracy:.2%}")

## Section 6: Temperature Scaling Effects

In [ ]:
# Demonstrate how temperature affects loss and confidence
temperatures = [0.01, 0.07, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0]
losses = []
batch_size = 30

for temp in temperatures:
    loss, _ = contrastive_loss(
        image_embeddings_proj[:batch_size], 
        text_embeddings_proj[:batch_size], 
        temperature=temp
    )
    losses.append(loss)

print("Temperature Effects on Contrastive Loss:")
for t, l in zip(temperatures, losses):
    print(f"  T={t:5.2f}: Loss={l:.4f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss vs temperature
ax1.plot(temperatures, losses, marker='o', linewidth=2.5, markersize=8, color='steelblue')
ax1.axvline(x=0.07, color='red', linestyle='--', linewidth=2, label='CLIP default (0.07)')
ax1.set_xlabel('Temperature', fontsize=11)
ax1.set_ylabel('Contrastive Loss', fontsize=11)
ax1.set_title('Effect of Temperature on Contrastive Loss', fontsize=12, fontweight='bold')
ax1.set_xscale('log')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# Softmax distributions for different temperatures
sample_logits = np.array([3.0, 1.0, 0.5, -0.5, -2.0])
for temp, color in [(0.07, 'red'), (0.5, 'orange'), (2.0, 'blue')]:
    probs = softmax(sample_logits / temp)
    ax2.plot(range(len(sample_logits)), probs, marker='o', label=f'T={temp}', linewidth=2, markersize=8)

ax2.set_xlabel('Class Index', fontsize=11)
ax2.set_ylabel('Probability', fontsize=11)
ax2.set_title('Softmax Distributions at Different Temperatures', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  Low T (e.g., 0.07): Sharp softmax → strong discrimination, higher loss for mistakes")
print("  High T (e.g., 2.0): Soft softmax → gentle gradients, lower loss")
print("  CLIP uses T=0.07 for sharp, confident matching.")

## Section 7: Cross-Modal Similarity and Modality Gap

In [ ]:
# Compute cross-modal similarity matrix
similarity_matrix = cosine_similarity(image_embeddings_proj, text_embeddings_proj)

print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Mean similarity (diagonal/matching pairs): {np.diag(similarity_matrix)[:min(10, len(similarity_matrix))].mean():.4f}")
print(f"Overall mean similarity: {similarity_matrix.mean():.4f}")
print(f"Overall std similarity: {similarity_matrix.std():.4f}")

# Compute recall metrics
def compute_recall_at_k(sim_matrix, k_values=[1, 3, 5, 10]):
    recalls = {}
    n = sim_matrix.shape[0]
    for k in k_values:
        if k > n:
            continue
        # For each image, check if correct text is in top-k
        top_k_indices = np.argsort(sim_matrix, axis=1)[:, -k:]
        matches = np.any(top_k_indices == np.arange(n).reshape(-1, 1), axis=1)
        recalls[f'Recall@{k}'] = matches.mean()
    return recalls

recalls = compute_recall_at_k(similarity_matrix)
print("\nCross-Modal Retrieval Performance:")
for metric, value in recalls.items():
    print(f"  {metric}: {value:.2%}")

# Visualize similarity matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap (subset for visibility)
subset_size = 20
sns.heatmap(similarity_matrix[:subset_size, :subset_size], ax=axes[0], cmap='coolwarm', 
            cbar_kws={'label': 'Cosine Similarity'}, vmin=-1, vmax=1)
axes[0].set_xlabel('Text Index')
axes[0].set_ylabel('Image Index')
axes[0].set_title('Cross-Modal Similarity (Subset)', fontweight='bold')

# Distribution of similarities
all_sims = similarity_matrix.flatten()
diag_sims = np.diag(similarity_matrix)
off_diag_sims = all_sims[~np.isin(np.arange(len(all_sims)), np.arange(len(similarity_matrix)) * similarity_matrix.shape[1] + np.arange(len(similarity_matrix)))]

axes[1].hist(off_diag_sims, bins=40, alpha=0.6, label='Unmatched pairs', color='blue')
axes[1].hist(diag_sims, bins=20, alpha=0.6, label='Matched pairs', color='red')
axes[1].set_xlabel('Cosine Similarity')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Similarities', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Section 8: Embedding Space Visualization (PCA)

In [ ]:
# Combine embeddings and reduce to 2D
combined_embeddings = np.vstack([image_embeddings_proj, text_embeddings_proj])
combined_modalities = ['Image'] * len(image_embeddings_proj) + ['Text'] * len(text_embeddings_proj)

# Extract concept labels from names
combined_concepts = [name.split('_')[0] + '_' + name.split('_')[1] for name in image_names] + \
                    [desc.split()[2] + '_' + desc.split()[3] for desc in text_descriptions]

# Apply PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(combined_embeddings)

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# By modality
for modality, color in [('Image', 'steelblue'), ('Text', 'coral')]:
    mask = np.array(combined_modalities) == modality
    axes[0].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                   alpha=0.6, s=60, color=color, label=modality, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[0].set_title('Embeddings Colored by Modality', fontweight='bold', fontsize=12)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# By concept
unique_concepts = sorted(set(combined_concepts))
colors_concept = plt.cm.tab10(np.arange(len(unique_concepts)) / len(unique_concepts))
concept_to_color = {c: colors_concept[i] for i, c in enumerate(unique_concepts)}

for concept in unique_concepts:
    mask = np.array(combined_concepts) == concept
    axes[1].scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1], 
                   alpha=0.6, s=60, color=concept_to_color[concept], label=concept, 
                   edgecolors='black', linewidth=0.5)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[1].set_title('Embeddings Colored by Concept', fontweight='bold', fontsize=12)
axes[1].legend(fontsize=9, ncol=2, loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Images and text should cluster together by concept")
print("- Modality gap visible: images and text may form separate clouds")
print("- Real models show better alignment than random embeddings")

## Section 9: Modality Gap Demonstration

In [ ]:
# Compute modality gap for each concept
gaps = []
concept_list = list(concepts.keys())

for concept in concept_list:
    # Find images and texts for this concept
    img_indices = [i for i, name in enumerate(image_names) if name.startswith(concept)]
    txt_indices = [i for i, desc in enumerate(text_descriptions) if concept.replace('_', ' ').split()[0].lower() in desc.lower()]
    
    if not img_indices or not txt_indices:
        continue
    
    # Mean embeddings per modality
    img_mean = image_embeddings_proj[img_indices].mean(axis=0)
    txt_mean = text_embeddings_proj[txt_indices].mean(axis=0)
    
    # Normalize
    img_mean = img_mean / np.linalg.norm(img_mean)
    txt_mean = txt_mean / np.linalg.norm(txt_mean)
    
    # Similarity
    sim = np.dot(img_mean, txt_mean)
    gap = 1 - sim
    gaps.append(gap)

print(f"Modality Gap Statistics:")
print(f"  Mean gap: {np.mean(gaps):.4f}")
print(f"  Std gap: {np.std(gaps):.4f}")
print(f"  Min gap: {np.min(gaps):.4f}")
print(f"  Max gap: {np.max(gaps):.4f}")

# Visualization
fig, ax = plt.subplots(figsize=(12, 5))
concept_names_plot = [c.replace('_', ' ').title() for c in concept_list[:len(gaps)]]
colors_bar = ['steelblue' if g < np.mean(gaps) else 'coral' for g in gaps]

ax.bar(range(len(gaps)), gaps, color=colors_bar, alpha=0.7, edgecolor='black', linewidth=1)
ax.axhline(y=np.mean(gaps), color='red', linestyle='--', linewidth=2, label=f'Mean gap: {np.mean(gaps):.4f}')
ax.set_xlabel('Concept', fontsize=11)
ax.set_ylabel('Modality Gap (1 - similarity)', fontsize=11)
ax.set_title('Modality Gap per Concept\n(How Much Do Image and Text Embeddings Differ?)', 
             fontweight='bold', fontsize=12)
ax.set_xticks(range(len(gaps)))
ax.set_xticklabels(concept_names_plot, rotation=45, ha='right')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Higher gap = image and text embeddings are more different")
print("- This is natural: vision and language capture different information")
print("- Alignment training (contrastive loss) reduces this gap")

## Section 10: Zero-Shot Classification

In [ ]:
# Use text embeddings as class prototypes
# Create class descriptions
class_descriptions = [
    "A red circle",
    "A blue square",
    "A green triangle",
    "A yellow pentagon",
    "A purple star"
]

# Encode class descriptions
class_embeddings = text_model.encode(class_descriptions, convert_to_tensor=True).cpu().numpy()
class_embeddings = class_embeddings / np.linalg.norm(class_embeddings, axis=1, keepdims=True)

# Test on a subset of images
test_subset_size = min(30, len(image_embeddings_np))
test_images = image_embeddings_np[:test_subset_size]
test_labels_zeroshot = [image_names[i].split('_')[0] + '_' + image_names[i].split('_')[1] for i in range(test_subset_size)]

# Classify by finding nearest class prototype
logits_zeroshot = np.dot(test_images, class_embeddings.T)
predictions_zeroshot = np.argmax(logits_zeroshot, axis=1)
pred_labels = [c.replace(' ', '_') for c in class_descriptions]

print("Zero-Shot Classification Examples:")
for i in range(min(10, test_subset_size)):
    true_class = test_labels_zeroshot[i].replace('_', ' ')
    pred_class = class_descriptions[predictions_zeroshot[i]]
    confidence = softmax(logits_zeroshot[i]).max()
    match = "✓" if true_class.lower() in pred_class.lower() or pred_class.lower() in true_class.lower() else "✗"
    print(f"  {match} True: {true_class:20s} | Pred: {pred_class:20s} | Conf: {confidence:.2%}")

# Compute overall accuracy
matches = sum(1 for i in range(test_subset_size) 
              if test_labels_zeroshot[i].split('_')[0] == class_descriptions[predictions_zeroshot[i]].split()[1].lower())
accuracy = matches / test_subset_size
print(f"\nZero-Shot Accuracy: {accuracy:.2%}")

## Section 11: Fusion Strategies - Early, Late, and Hybrid

In [ ]:
# Simulate multimodal classification with different fusion strategies
n_samples = 50
n_classes = 5

# Synthetic logits from vision and text models
np.random.seed(42)
vision_logits = np.random.randn(n_samples, n_classes) * 2
vision_logits[np.arange(n_samples), np.arange(n_samples) % n_classes] += 3.5

text_logits = np.random.randn(n_samples, n_classes) * 2
text_logits[np.arange(n_samples), np.arange(n_samples) % n_classes] += 3.0

y_true = np.arange(n_samples) % n_classes

# Fusion strategies
# 1. Early fusion (average logits before softmax)
early_fusion = (vision_logits + text_logits) / 2
pred_early = np.argmax(early_fusion, axis=1)
acc_early = (pred_early == y_true).mean()

# 2. Late fusion (average probabilities)
vision_probs = softmax(vision_logits, axis=1)
text_probs = softmax(text_logits, axis=1)
late_fusion = (vision_probs + text_probs) / 2
pred_late = np.argmax(late_fusion, axis=1)
acc_late = (pred_late == y_true).mean()

# 3. Hybrid fusion (weighted combination)
w_vision, w_text = 0.6, 0.4
hybrid_fusion = w_vision * vision_probs + w_text * text_probs
pred_hybrid = np.argmax(hybrid_fusion, axis=1)
acc_hybrid = (pred_hybrid == y_true).mean()

# Unimodal baselines
pred_vision = np.argmax(vision_logits, axis=1)
acc_vision = (pred_vision == y_true).mean()

pred_text = np.argmax(text_logits, axis=1)
acc_text = (pred_text == y_true).mean()

print("Fusion Strategy Comparison:")
print(f"  Vision only:              {acc_vision:.2%}")
print(f"  Text only:                {acc_text:.2%}")
print(f"  Early fusion:             {acc_early:.2%}")
print(f"  Late fusion:              {acc_late:.2%}")
print(f"  Hybrid fusion (0.6/0.4):  {acc_hybrid:.2%}")

# Visualization
fig, ax = plt.subplots(figsize=(11, 6))
strategies = ['Vision\nOnly', 'Text\nOnly', 'Early\nFusion', 'Late\nFusion', 'Hybrid\nFusion']
accuracies = [acc_vision, acc_text, acc_early, acc_late, acc_hybrid]
colors_strategy = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

bars = ax.bar(strategies, accuracies, color=colors_strategy, alpha=0.8, edgecolor='black', linewidth=1.5, width=0.6)
ax.set_ylabel('Classification Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Multimodal Fusion Strategies Comparison', fontsize=13, fontweight='bold')
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
           f'{acc:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Section 12: Multimodal Architecture Comparison

In [ ]:
architecture_data = {
    'Model': [
        'CLIP',
        'ViLBERT',
        'ALBEF',
        'BLIP',
        'Flamingo',
        'LLaVA'
    ],
    'Architecture Type': [
        'Contrastive Learning',
        'Multimodal Fusion',
        'Alignment + Contrastive',
        'Encoder-Decoder',
        'LLM + Vision Adapter',
        'LLM + Vision Encoder'
    ],
    'Vision Encoder': [
        'Vision Transformer',
        'ResNet',
        'Vision Transformer',
        'Vision Transformer',
        'Perceiver Resampler',
        'CLIP ViT'
    ],
    'Text Model': [
        'Transformer',
        'BERT',
        'BERT',
        'BERT',
        'Flamingo LLM',
        'LLaMA'
    ],
    'Primary Task': [
        'Image-Text Retrieval',
        'Multimodal Understanding',
        'Cross-Modal Alignment',
        'Image Captioning',
        'Visual Question Answering',
        'Visual Instruction Following'
    ],
    'Key Advantage': [
        'Efficient zero-shot',
        'Early fusion benefits',
        'Strong alignment',
        'Flexible generation',
        'Few-shot learning',
        'Instruction following'
    ]
}

arch_df = pd.DataFrame(architecture_data)

print("\n" + "="*140)
print("MULTIMODAL AI ARCHITECTURE COMPARISON")
print("="*140)
print(arch_df.to_string(index=False))
print("="*140)

## Section 13: Key Concepts Summary

In [ ]:
summary_text = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║                 MULTIMODAL AI: KEY CONCEPTS SUMMARY                            ║
╚═══════════════════════════════════════════════════════════════════════════════╝

1. CONTRASTIVE LEARNING (InfoNCE / CLIP-style)
   ─────────────────────────────────────────────
   • Objective: Match image-text pairs with high similarity
   • Unmatched pairs pushed apart in embedding space
   • Cross-entropy loss over batch of paired samples
   • Temperature τ controls confidence (0.07 in CLIP)

2. EMBEDDINGS & SHARED SPACE
   ──────────────────────────
   • Each modality encoded independently
   • Projected to common dimension (e.g., 256-d)
   • Normalized to unit norm (L2 normalization)
   • Cosine similarity measures cross-modal alignment

3. MODALITY GAP
   ─────────────
   • Vision and text naturally diverge (different sensors)
   • Image: spatial, color, texture information
   • Text: semantic, relational, abstract concepts
   • Alignment training narrows but doesn't eliminate gap

4. FUSION STRATEGIES
   ──────────────────
   Early Fusion:  Combine embeddings before classification
                  → Better: computational efficiency
   
   Late Fusion:   Combine predictions/probabilities
                  → Better: modality independence
   
   Hybrid Fusion: Weighted combination of both
                  → Best: balances efficiency and flexibility

5. ZERO-SHOT CLASSIFICATION
   ─────────────────────────
   • Classify images using text descriptions
   • No training on target classes required
   • Text embeddings = class prototypes
   • Enables transfer to unseen classes

6. RETRIEVAL EVALUATION
   ────────────────────
   Recall@K: % of queries where correct match is in top-K
   - Recall@1: Is the best match correct?
   - Recall@5: Is the correct match in top 5?
   - Used for image-text retrieval, re-ranking

7. TEMPERATURE SCALING
   ────────────────────
   • Low T (0.01-0.1):  Sharp softmax → strong discrimination
   • High T (1.0-5.0):   Soft softmax → gentle gradients
   • Controls trade-off between confidence and generalization
   • CLIP uses T=0.07 for stable training

8. MODERN ARCHITECTURES
   ─────────────────────
   CLIP:        Contrastive learning, efficient retrieval
   ViLBERT:     Early fusion, joint pre-training
   ALBEF:       Alignment + contrastive, strong caption
   BLIP:        Encoder-decoder, flexible generation
   Flamingo:    LLM + vision, in-context learning
   LLaVA:       Chat-based, instruction following

═══════════════════════════════════════════════════════════════════════════════
"""

print(summary_text)

## Interview Takeaways

In [ ]:
interview_tips = """
╔═══════════════════════════════════════════════════════════════════════════════╗
║                    INTERVIEW PREPARATION TIPS                                 ║
╚═══════════════════════════════════════════════════════════════════════════════╝

📋 COMMON INTERVIEW QUESTIONS:

1. "Explain contrastive learning in multimodal systems."
   Answer:
   - Uses InfoNCE loss to align embeddings from different modalities
   - Positive pairs (matched image-text) pushed together
   - Negative pairs (unmatched) pushed apart
   - Temperature τ controls softness of softmax
   - Efficiently learns shared representation space

2. "What is the modality gap and how do you address it?"
   Answer:
   - Vision and text naturally diverge (different sensors)
   - Cross-modal alignment training narrows the gap
   - Fusion strategies (early, late, hybrid) handle residual gap
   - More training data and model scale help
   - Hard negatives sampling improves alignment

3. "Compare early vs. late fusion."
   Answer:
   Early:  Fuse embeddings/features → computationally efficient, but
           modalities must be compatible
   Late:   Fuse predictions → more flexible, preserves modality
           independence, but requires two forward passes
   Hybrid: Weighted combination of both

4. "How would you evaluate a multimodal model?"
   Answer:
   - Recall@K for retrieval tasks (image-to-text, text-to-image)
   - Classification accuracy for recognition
   - BLEU/CIDEr scores for captioning
   - Zero-shot performance on held-out classes
   - Alignment metrics (cosine similarity of matched pairs)

5. "What makes CLIP effective?"
   Answer:
   - Contrastive learning is efficient and scalable
   - Learns from web-scale image-text pairs
   - Strong zero-shot transfer (no fine-tuning needed)
   - Simple architecture (ResNet/ViT + Transformer)
   - Temperature-scaled softmax stabilizes training

6. "How do you handle class imbalance in multimodal learning?"
   Answer:
   - Hard negative mining (sample most confusing negatives)
   - Class-balanced sampling during data loading
   - Focal loss variants for emphasis on hard examples
   - Oversampling underrepresented classes
   - Contrastive learning naturally handles imbalance better than softmax

7. "Explain zero-shot classification using text embeddings."
   Answer:
   - Text descriptions of unseen classes → embeddings (prototypes)
   - New image → embedding via vision encoder
   - Classify by finding nearest class prototype
   - No training on target classes required
   - Enables transfer to arbitrary new classes

💡 INDUSTRY INSIGHTS:

- CLIP's success showed the power of contrastive learning at scale
- Vision-language models enable new applications (VQA, captioning, retrieval)
- Alignment is harder than single-modality learning (≥2x harder)
- Larger models and datasets dramatically improve cross-modal transfer
- Temperature scheduling can improve convergence (increase T over time)
- Open-source models (OpenCLIP) make these techniques accessible

🎯 PRACTICAL TIPS FOR INTERVIEWS:

1. Mention specific papers (CLIP, ViLBERT, BLIP, LLaVA)
2. Show understanding of why contrastive loss works
3. Discuss trade-offs between different architectures
4. Know evaluation metrics (Recall@K, BLEU, etc.)
5. Relate concepts to real-world use cases
6. Ask clarifying questions about deployment constraints

═══════════════════════════════════════════════════════════════════════════════
"""

print(interview_tips)

---

<small><em>© 2026 AI Nirvana · More Info: https://medium.com/@snigam/a-simple-structured-way-to-prepare-for-ai-ml-interviews-68b2e5830195 · Disclaimer: Provided as is. No liability assumed.</em></small>